# Supply chain: a reproducible data-quality analysis

This notebook uses only `cleaned_supply_chain_dataset.csv` in this repository. The dataset is simulated. Its name suggests clean data, but most order dates are absent. Accordingly, this analysis focuses on completeness and category costs; it does **not** estimate monthly trends or delivery speed from the incomplete dates. Run from the repository root.


In [ ]:
import pandas as pd
from pathlib import Path

source = Path("cleaned_supply_chain_dataset.csv")
df = pd.read_csv(source)
assert len(df) == 3000
print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
print(df.dtypes.to_string())


In [ ]:
# Audit required columns before interpreting the old dashboard.
issues = pd.Series({
    "Missing order dates": df["order_date"].isna().sum(),
    "Missing delivery dates": df["delivery_date"].isna().sum(),
    "Cost mismatches (> 0.02)": ((df["total_cost"] - df["quantity"] * df["unit_price"]).abs() > .02).sum(),
    "Missing product defect values": df["product_defects"].isna().sum(),
})
print(issues.to_string())
print("Complete date pairs:", df[["order_date", "delivery_date"]].notna().all(axis=1).sum())
assert issues["Missing order dates"] > 0, "Revisit the scope if a complete source is supplied."


In [ ]:
# Descriptive cost totals use complete cost fields; currency is unspecified in the source.
category = df.groupby("product_category", dropna=False).agg(
    orders=("order_id", "size"),
    recorded_cost=("total_cost", "sum")
).sort_values("recorded_cost", ascending=False)
category["share_of_recorded_cost"] = category["recorded_cost"] / category["recorded_cost"].sum()
print(category.round(3).to_string())


In [ ]:
# Standardise inconsistent supplier spelling without changing the source file.
working = df.copy()
working["supplier_clean"] = working["supplier_name"].str.extract(r"(?i)^supplier\s*([a-e])$")[0].str.upper().radd("Supplier ")
print("Supplier labels before:", df["supplier_name"].nunique())
print("Supplier labels after:", working["supplier_clean"].nunique())
print(working["supplier_clean"].value_counts().to_string())


## Interpretation and next steps

The cost comparison is descriptive. First investigate why 2,683 order dates and 296 delivery dates are missing and why some recorded costs disagree with quantity × unit price. Do not fill missing dates with invented values. If the original raw export becomes available, compare it to this file and then rerun time-based analysis using documented exclusions. The screenshots in `archive/legacy-visuals/` come from an earlier notebook and are retained for provenance, not used as verified findings here.
